# 03 - Multi-agent: routing e ruoli

Un sistema multi-agent non deve essere teatrale. Qui ogni agente ha un ruolo piccolo: un router sceglie chi deve lavorare, uno specialista raccoglie fatti, un writer scrive la risposta finale.

La cosa interessante per gli studenti è il trace: si vede quale strada ha preso la richiesta e dove correggere il sistema.

## Obiettivi
- scomporre un assistente in ruoli
- costruire un router leggibile
- mantenere una traccia dei passaggi
- aggiungere o modificare uno specialista senza riscrivere tutto


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "lab_agentic").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import re
from typing import TypedDict

from lab_agentic.utils import (
    find_event,
    format_course,
    format_event,
    format_room,
    get_chat_model,
    get_course,
    keyword_router,
    office_hours_lookup,
    policy_lookup,
    room_recommendation,
    schedule_conflict,
    search_courses,
)
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph


In [2]:
PROVIDER = "ollama_cloud"        # prova anche: "ollama_cloud"
MODEL_NAME = None

llm = get_chat_model(provider=PROVIDER, model=MODEL_NAME, temperature=0)


## Stato condiviso del team

Lo stato passa da un nodo all'altro. Tenere visibili `route`, `facts`, `answer` e `trace` rende il sistema spiegabile.


In [3]:
class TeamState(TypedDict, total=False):
    question: str
    route: str
    facts: str
    answer: str
    trace: list[str]


## Router

Questo router è deterministico di proposito. È meno elegante di un router LLM, ma è molto più facile da capire.


In [4]:
def route_question(question: str) -> str:
    q = question.lower()
    if any(token in q for token in ["evento", "demo", "escape", "workshop", "clinic", "portfolio", "pizza"]):
        return "events"
    return keyword_router(question)


def router_agent(state: TeamState):
    route = route_question(state["question"])
    return {
        "route": route,
        "trace": state.get("trace", []) + [f"router -> {route}"],
    }


for question in [
    "Quale corso insegna LangGraph?",
    "Mi serve un'aula con registrazione per 35 persone.",
    "AI301 e SE220 si sovrappongono?",
    "Qual è la policy sulle consegne in ritardo?",
    "Quando c'è la Pizza Review sul testing?",
]:
    print(question, "=>", route_question(question))


Quale corso insegna LangGraph? => catalog
Mi serve un'aula con registrazione per 35 persone. => room
AI301 e SE220 si sovrappongono? => schedule
Qual è la policy sulle consegne in ritardo? => policy
Quando c'è la Pizza Review sul testing? => events


## Specialisti

Ogni specialista raccoglie fatti. Solo il writer chiede al modello di scrivere la risposta finale.


In [5]:
def add_trace(state: TeamState, message: str) -> list[str]:
    return state.get("trace", []) + [message]


def search_terms(question: str) -> list[str]:
    stopwords = {
        "cosa", "quale", "quando", "dove", "serve", "corso", "aula",
        "evento", "trova", "dimmi", "studenti", "persone", "della", "dello",
    }
    terms = re.findall(r"[A-Za-zÀ-ÿ][A-Za-zÀ-ÿ0-9-]+", question.lower())
    return [term for term in terms if len(term) >= 4 and term not in stopwords]


def catalog_agent(state: TeamState):
    codes = re.findall(r"[A-Z]{2,}\d{3}", state["question"].upper())
    if codes:
        rows = [get_course(code) for code in codes]
        facts = "\n\n".join(format_course(row) for row in rows if row)
    else:
        rows = []
        for term in search_terms(state["question"]):
            rows = search_courses(term)
            if rows:
                break
        facts = "\n\n".join(format_course(row) for row in rows[:3])
    return {"facts": facts or "Nessun corso trovato.", "trace": add_trace(state, "catalog_agent")}


def room_agent(state: TeamState):
    numbers = [int(n) for n in re.findall(r"\d+", state["question"])]
    min_capacity = max(numbers) if numbers else 1
    needs_recording = "registr" in state["question"].lower() or "record" in state["question"].lower()
    rows = room_recommendation(min_capacity=min_capacity, needs_recording=needs_recording)
    facts = "\n".join(format_room(row) for row in rows)
    return {"facts": facts or "Nessuna aula trovata.", "trace": add_trace(state, "room_agent")}


def schedule_agent(state: TeamState):
    codes = re.findall(r"[A-Z]{2,}\d{3}", state["question"].upper())
    if len(codes) >= 2:
        facts = str(schedule_conflict(codes[0], codes[1]))
    else:
        facts = office_hours_lookup(state["question"])
    return {"facts": facts, "trace": add_trace(state, "schedule_agent")}


def policy_agent(state: TeamState):
    facts = policy_lookup(state["question"])
    return {"facts": facts, "trace": add_trace(state, "policy_agent")}


def events_agent(state: TeamState):
    rows = []
    for term in search_terms(state["question"]):
        rows = find_event(term)
        if rows:
            break
    facts = "\n".join(format_event(row) for row in rows)
    return {"facts": facts or "Nessun evento trovato.", "trace": add_trace(state, "events_agent")}


def general_agent(state: TeamState):
    facts = "Nessuno specialista ha riconosciuto la domanda. Serve una risposta generale e onesta su cosa manca."
    return {"facts": facts, "trace": add_trace(state, "general_agent")}


In [6]:
def writer_agent(state: TeamState):
    content = "\n".join([
        f"Domanda: {state['question']}",
        f"Route scelta: {state.get('route')}",
        "Fatti dallo specialista:",
        state.get("facts", ""),
        "",
        "Scrivi la risposta finale in italiano, breve e utile.",
    ])
    prompt = [
        SystemMessage(
            content=(
                "Sei il writer finale di un team multi-agent universitario. "
                "Usa solo i fatti forniti quando sono fattuali. "
                "Se mancano dati, dillo chiaramente."
            )
        ),
        HumanMessage(content=content),
    ]
    answer = llm.invoke(prompt).content
    return {"answer": answer, "trace": add_trace(state, "writer_agent")}


## Costruisci il grafo

L'arco condizionale collega la route allo specialista giusto.


In [7]:
def pick_worker(state: TeamState) -> str:
    return state["route"]


builder = StateGraph(TeamState)
builder.add_node("router", router_agent)
builder.add_node("catalog_agent", catalog_agent)
builder.add_node("room_agent", room_agent)
builder.add_node("schedule_agent", schedule_agent)
builder.add_node("policy_agent", policy_agent)
builder.add_node("events_agent", events_agent)
builder.add_node("general_agent", general_agent)
builder.add_node("writer_agent", writer_agent)

builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    pick_worker,
    {
        "catalog": "catalog_agent",
        "room": "room_agent",
        "schedule": "schedule_agent",
        "policy": "policy_agent",
        "events": "events_agent",
        "writer": "general_agent",
    },
)
for worker in ["catalog_agent", "room_agent", "schedule_agent", "policy_agent", "events_agent", "general_agent"]:
    builder.add_edge(worker, "writer_agent")
builder.add_edge("writer_agent", END)

team = builder.compile()


## Esegui il team

Guarda prima la route e il trace, poi la risposta. Se la risposta è sbagliata, spesso il trace dice dove intervenire.


In [8]:
def run_team(question: str):
    result = team.invoke({"question": question, "trace": []})
    print("ROUTE:", result["route"])
    print("TRACE:", " -> ".join(result["trace"]))
    print("FATTI:")
    print(result["facts"])
    print()
    print("RISPOSTA:")
    print(result["answer"])
    return result

run_team("AI301 e SE220 si sovrappongono di orario?")


ROUTE: schedule
TRACE: router -> schedule -> schedule_agent -> writer_agent
FATTI:
{'conflict': True, 'overlap_slots': ['Wed 10:00-11:00'], 'reason': 'sovrapposizione trovata'}

RISPOSTA:
Sì, i corsi AI301 e SE220 si sovrappongono di orario il mercoledì dalle 10:00 alle 11:00.


{'question': 'AI301 e SE220 si sovrappongono di orario?',
 'route': 'schedule',
 'facts': "{'conflict': True, 'overlap_slots': ['Wed 10:00-11:00'], 'reason': 'sovrapposizione trovata'}",
 'answer': 'Sì, i corsi AI301 e SE220 si sovrappongono di orario il mercoledì dalle 10:00 alle 11:00.',
 'trace': ['router -> schedule', 'schedule_agent', 'writer_agent']}

In [9]:
play_questions = [
    "Quale corso insegna LangGraph e tool use?",
    "Mi serve un'aula con registrazione per 35 studenti.",
    "Qual è la rubrica di valutazione?",
    "Quando c'è l'AI Escape Room?",
    "Aiutami a scegliere un corso se mi piacciono chatbot e retrieval.",
]

for question in play_questions:
    print()
    print("=" * 80)
    print("DOMANDA:", question)
    run_team(question)



DOMANDA: Quale corso insegna LangGraph e tool use?
ROUTE: catalog
TRACE: router -> catalog -> catalog_agent -> writer_agent
FATTI:
AI301 — Sistemi Agentici per il Campus
Docente: Dr.ssa Sofia Bianchi
Orario: Mon 10:00-12:00, Wed 10:00-11:00
Aula: A201
Parole chiave: agenti, llm, langgraph, strumenti, tool use, planning, campus

RISPOSTA:
Il corso che insegna LangGraph e il tool use è **AI301 — Sistemi Agentici per il Campus**, tenuto dalla Dr.ssa Sofia Bianchi.

DOMANDA: Mi serve un'aula con registrazione per 35 studenti.
ROUTE: room
TRACE: router -> room -> room_agent -> writer_agent
FATTI:
A201: capienza 40; dotazione proiettore,lavagna,registrazione
B110: capienza 60; dotazione proiettore,registrazione,microfoni
Aula-Magna: capienza 120; dotazione palco,proiettore,registrazione,microfoni

RISPOSTA:
Ecco le opzioni disponibili per un'aula con registrazione per 35 studenti:

*   **A201**: capienza 40 (dotata di proiettore, lavagna e registrazione).
*   **B110**: capienza 60 (dotata d

## Sfida

Scegli una modifica e osserva il trace:

- aggiungi route `career` per domande su CV e portfolio
- fai riconoscere meglio le domande su ricevimento
- sostituisci il router deterministico con un router LLM
- aggiungi un agente di sicurezza che intercetta email, prenotazioni e voti
